# Problems 1 and 2: Reconciliation and stuck orders

**Problem 1:** Each order is stored in several tables (orders, items, payments, reviews). If they don't match, revenue numbers are wrong and refunds can be missed. I want to find which orders don't match, why, and what would prevent it.

**Problem 2:** 1,729 orders never reached a final status (delivered or canceled), even though the customer already paid. I need a way to prioritise them so a small team can close them, and a rule so the backlog doesn't build up again.

Most of the logic is in SQL (`sql/01_reconciliation.sql` and `sql/02_stuck_orders.sql`). Here I look for patterns, make the charts and build the worklist.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import chart_style as cs
cs.apply(); warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
con = sqlite3.connect(ROOT / "data" / "olist.db")
OUT = ROOT / "data" / "processed"

## Problem 1: Reconciliation checks

In [2]:
summ = pd.read_csv(OUT / "recon_summary.csv")
summ

,check_id,orders_failing,pct_of_orders,amount_at_stake_brl,description
0,C1_order_without_items,775,0.78,"162,591.95",Payment recorded but no line items: cannot be ...
1,C2_order_without_payment,1,0.00,143.46,Line items exist but no payment record
2,C3_overpaid_vs_basket,291,0.29,"3,080.17",Collected more than items + freight
3,C4_underpaid_vs_basket,141,0.14,"4,899.23",Collected less than items + freight
4,C5_delivered_without_delivery_date,8,0.01,"1,378.92",Status = delivered but no customer delivery ti...
5,C6_delivered_without_approval,14,0.01,"1,954.60",Status = delivered but payment approval timest...
6,C7_carrier_before_purchase,166,0.17,"23,130.02",Carrier pickup timestamp earlier than purchase
7,C8_customer_before_carrier,23,0.02,"4,536.71",Customer delivery earlier than carrier pickup
8,C9_canceled_but_delivered,6,0.01,748.58,Status = canceled but a customer delivery date...


In [3]:
lab = {"C1_order_without_items": "C1 Paid, but no line items", "C2_order_without_payment": "C2 Items, but no payment",
       "C3_overpaid_vs_basket": "C3 Paid more than basket", "C4_underpaid_vs_basket": "C4 Paid less than basket",
       "C5_delivered_without_delivery_date": "C5 'Delivered' with no date", "C6_delivered_without_approval": "C6 'Delivered' never approved",
       "C7_carrier_before_purchase": "C7 Carrier pickup before purchase", "C8_customer_before_carrier": "C8 Delivered before pickup",
       "C9_canceled_but_delivered": "C9 Canceled, yet delivered"}
p = summ.assign(label=summ.check_id.map(lab)).sort_values("orders_failing")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, col, color, fmt, ttl in [(axes[0], "orders_failing", cs.PLUM, "{:,.0f}", "Orders failing"),
                                 (axes[1], "amount_at_stake_brl", cs.ORANGE, "R$ {:,.0f}", "Money involved (BRL)")]:
    ax.barh(p.label, p[col], color=color, height=0.6)
    for i, v in enumerate(p[col]): ax.text(v, i, "  " + fmt.format(v), va="center", fontsize=8.5, color=cs.GREY)
    ax.set_xlim(0, p[col].max() * 1.35); ax.grid(axis="y", visible=False); ax.set_xlabel(ttl)
from matplotlib.ticker import FuncFormatter
axes[1].xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
c1 = summ.set_index("check_id").loc["C1_order_without_items"]
fig.suptitle("Reconciliation checks", fontweight="bold", fontsize=12)
cs.save(fig, "p1_recon_checks")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p1_recon_checks.png')

### C1: orders with a payment but no items

From `recon_no_items_by_status`: 99% of `unavailable` orders and 26% of `canceled` orders have no items, but no delivered order is missing items. It looks like the items get removed when the product is out of stock, but the payment record stays. The dataset has no refund table, so I can't confirm whether these R$ 162.6k were refunded. That's something finance should check first.

In [4]:
pd.read_csv(OUT / "recon_no_items_by_status.csv")

,order_status,orders,orders_without_items,pct_without_items
0,unavailable,609,603,99.00
1,canceled,625,164,26.20
2,created,5,5,100.00
3,invoiced,314,2,0.60
4,shipped,1107,1,0.10
5,processing,301,0,0.00
6,delivered,96478,0,0.00
7,approved,2,0,0.00


### C3: overpayments increase with instalments

If this was just random data entry errors, it wouldn't depend on the number of instalments. But it goes up steadily with instalments, which points to instalment interest (*parcelamento com juros*) being included in the payment without its own field. So this needs a schema change, not a data clean-up.

In [5]:
ins = pd.read_csv(OUT / "recon_mismatch_by_installments.csv")
fig, ax = plt.subplots(figsize=(10, 3.8))
xx = np.arange(len(ins))
b = ax.bar(xx, ins.overpaid_pct, color=cs.MUSTARD, width=0.55)
for i, r_ in ins.iterrows():
    ax.text(i, r_.overpaid_pct + 0.03, f"{r_.overpaid_pct:.2f}% of orders\navg +{r_.avg_overpay_pct_of_basket:.1f}% of basket",
            ha="center", fontsize=8.5, color=cs.GREY)
ax.set_xticks(xx, [f"{b_} instalments" for b_ in ins.installment_band]); ax.grid(axis="x", visible=False)
ax.set_ylabel("% of orders overpaid"); ax.set_ylim(0, 1.25)
ax.set_title("Overpaid orders by number of instalments")
cs.save(fig, "p1_overpay_by_installments")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p1_overpay_by_installments.png')

### C7: carrier pickup earlier than purchase

All 166 cases are from April 2018 or later, and the median gap is only ~36 minutes. A parcel can't be picked up before it's bought, so this is most likely a change in how the carrier timestamp was recorded (time zone or batch time) from April 2018. It also affects the seller handling time KPI after that date.

In [6]:
c7 = pd.read_sql("""SELECT order_id, order_purchase_timestamp AS purchased, order_delivered_carrier_date AS carrier
                    FROM orders WHERE order_delivered_carrier_date < order_purchase_timestamp""", con)
c7["gap_hours"] = (pd.to_datetime(c7.purchased) - pd.to_datetime(c7.carrier)).dt.total_seconds() / 3600
c7["month"] = c7.purchased.str[:7]
by_m = c7.groupby("month").size().reindex(pd.read_csv(OUT/"monthly_kpis.csv").purchase_month, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(range(len(by_m)), by_m.values, color=[cs.ORANGE if v else cs.GRIDC for v in by_m.values], width=0.6)
ax.set_xticks(range(len(by_m)), [m[2:] for m in by_m.index], fontsize=8.5); ax.grid(axis="x", visible=False)
ax.set_ylabel("orders")
ax.set_title("Orders where carrier pickup is before purchase (C7), by month")
cs.save(fig, "p1_c7_timeline")
c7.gap_hours.describe(percentiles=[.25, .5, .75, .95]).round(2)

count     166.00
mean       26.04
std       318.92
min         0.01
25%         0.23
50%         0.60
75%         1.02
95%         1.82
max     4,109.10
Name: gap_hours, dtype: float64

### Summary: root causes and controls

| Check | Likely cause | Prevent | Detect |
|---|---|---|---|
| C1 paid, no items (775 orders, R$ 162.6k) | Items removed on stock-out, payment kept | Stock-out should trigger a refund and keep the item line as `void` | Daily query: unavailable orders with payment and no refund, sent to finance |
| C3/C4 amount mismatch (432) | Instalment interest not stored; partial voucher use | Add `interest_amount` and `discount_amount` to payments | Flag orders where the gap is more than R$ 1 after interest |
| C7/C8 impossible timestamps (189) | Carrier timestamp change from Apr 2018 | Carrier integration should send UTC; reject events before purchase time | Track % of orders with negative stage time (should be 0) |
| C5/C6/C9 status vs dates (28) | Manual status changes | Only allow status changes through events | Weekly exception report |

## Problem 2: Stuck orders

In [7]:
st = pd.read_csv(OUT / "stuck_orders.csv")
pd.read_csv(OUT / "stuck_summary.csv")

,bucket,orders,value_brl,avg_value_brl,avg_age_days,past_promise,sellers_involved
0,B1_lost_in_carrier_network,1107,"177,207.07",160.08,315.00,1107,498
1,B2_seller_never_shipped,615,"138,532.09",225.26,383.00,615,346
2,B3_never_released_to_seller,7,929.18,132.74,386.00,7,2


In [8]:
bm = (st.groupby(["purchase_month", "bucket"]).size().unstack(fill_value=0)
        .reindex(pd.read_csv(OUT/"monthly_kpis.csv").purchase_month, fill_value=0))
fig, ax = plt.subplots(figsize=(11, 4.2))
bottom = np.zeros(len(bm))
for col, color, lab_ in [("B1_lost_in_carrier_network", cs.GREEN, "B1 Shipped, never delivered"),
                         ("B2_seller_never_shipped", cs.ORANGE, "B2 Invoiced, seller never shipped"),
                         ("B3_never_released_to_seller", cs.MUSTARD, "B3 Never released to seller")]:
    if col in bm:
        ax.bar(range(len(bm)), bm[col], bottom=bottom, color=color, label=lab_, width=0.65, edgecolor="white", linewidth=1.5)
        bottom += bm[col].values
ax.set_xticks(range(len(bm)), [m[2:] for m in bm.index], fontsize=8.5); ax.grid(axis="x", visible=False)
ax.set_ylabel("orders still open, by purchase month"); ax.legend(loc="upper left")
ax.set_title("Stuck orders by purchase month")
cs.save(fig, "p2_backlog_inflow")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p2_backlog_inflow.png')

In [9]:
conc = pd.read_csv(OUT / "stuck_seller_concentration.csv")
top10 = conc.loc[conc.seller_rank == 10, "cum_pct_of_backlog"].item()
print(f"Sellers involved: {len(conc):,} | Top-10 sellers hold only {top10:.1f}% of stuck orders")
print(f"Orders past promise date: {(st.days_past_promise > 0).mean():.1%} | median age: {st.age_days.median():.0f} days")

Sellers involved: 751 | Top-10 sellers hold only 12.7% of stuck orders
Orders past promise date: 100.0% | median age: 322 days


The backlog is spread over 751 sellers (top 10 only have ~13%), so calling sellers one by one won't work. I bucket the orders by where they stopped, then prioritise by value, and batch the work by seller or carrier.

### Priority rules

| Priority | Rule | Reason |
|---|---|---|
| P1 | Value >= R$ 200 | Highest refund / chargeback risk, worth a manual trace |
| P2 | R$ 80 - 200 | Normal handling, batched per seller (B2) or carrier (B1) |
| P3 | Value < R$ 80, or bucket B3 | Handling costs more than the order: auto-cancel, refund and send a voucher |

Within each priority, oldest orders first.

In [10]:
def priority(r_):
    if r_.bucket == "B3_never_released_to_seller" or r_.order_value < 80: return "P3"
    return "P1" if r_.order_value >= 200 else "P2"

ACTION = {"B1_lost_in_carrier_network": "Open carrier trace / lost-parcel claim; refund customer if not found in 5 days",
          "B2_seller_never_shipped":    "Contact seller (batched per seller): ship within 48h or cancel + refund; strike on scorecard",
          "B3_never_released_to_seller":"Auto-cancel and refund (payment never cleared to seller)"}
st["priority"] = st.apply(priority, axis=1)
st["auto_resolvable"] = (st.priority == "P3").astype(int)
st["recommended_action"] = st.bucket.map(ACTION)
st.loc[st.priority == "P3", "recommended_action"] = "Auto-cancel + refund + apology voucher (rule-based, no agent touch)"
st["seller_batch_size"] = st.groupby("seller_id").order_id.transform("count")
st = st.sort_values(["priority", "bucket", "age_days"], ascending=[True, True, False])
st["work_rank"] = range(1, len(st) + 1)
st.to_csv(OUT / "stuck_orders_prioritized.csv", index=False)

pv = st.pivot_table(index="priority", columns="bucket", values="order_id", aggfunc="count", fill_value=0)
pv["total_orders"] = pv.sum(axis=1)
pv["value_brl"] = st.groupby("priority").order_value.sum().round(0)
pv

bucket,B1_lost_in_carrier_network,B2_seller_never_shipped,B3_never_released_to_seller,total_orders,value_brl
priority,,,,,
P1,219,204,0,423,"192,678.00"
P2,469,236,0,705,"91,846.00"
P3,419,175,7,601,"32,144.00"


In [11]:
agg = st.groupby("priority").agg(orders=("order_id", "count"), value=("order_value", "sum"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"wspace": 0.35})
cols = [cs.BRICK, cs.ORANGE, cs.SAGE]
for ax, col, ttl, fmt in [(axes[0], "orders", "Orders", "{:,.0f}"), (axes[1], "value", "Value (BRL)", "R$ {:,.0f}")]:
    ax.barh(agg.index[::-1], agg[col][::-1], color=cols[::-1], height=0.55)
    for i, v in enumerate(agg[col][::-1]): ax.text(v, i, "  " + fmt.format(v), va="center", fontsize=9, color=cs.GREY)
    ax.set_xlim(0, agg[col].max() * 1.4); ax.grid(axis="y", visible=False); ax.set_xlabel(ttl)
from matplotlib.ticker import FuncFormatter
axes[1].xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
fig.suptitle("Backlog by priority", fontweight="bold", fontsize=12)
pct_auto = agg.loc["P3", "orders"] / agg.orders.sum()
cs.save(fig, "p2_priority_split")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p2_priority_split.png')

### Stopping the backlog from coming back

1. A nightly job (about 2 hours of engineering work) that flags orders still not delivered 7 days after the promise date: B3 is auto-cancelled, B2 sellers get a "ship in 48h or cancel" message, B1 opens a carrier trace.
2. Every B2 order counts as a strike on the seller scorecard (used in Problem 4).
3. Weekly KPIs: open orders past promise date (from 1,729 to under 50), median age of open orders (under 10 days), % closed automatically.

The time needed to clear P1 and P2 is calculated in the Excel file (`Capacity_Model` sheet) using the worklist exported above.